# Step 6 + SBERT-FT quick fusion

Notebook này không train và không inference lại model. Nó dùng lại Step 6 public ranking và SBERT-FT public ranking đã chạy xong, sau đó tạo `submission.zip` để nộp thử.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import zipfile
from collections import OrderedDict
from pathlib import Path
from typing import Any

MAX_SUBMISSION_DOCS = 5
RRF_K = 60
DEFAULT_CANDIDATE = 'rrf_sbert0p60'
FUSION_CANDIDATES = OrderedDict({
    'rrf_sbert0p25': {'kind': 'rrf', 'step6_weight': 1.0, 'sbert_weight': 0.25},
    'rrf_sbert0p40': {'kind': 'rrf', 'step6_weight': 1.0, 'sbert_weight': 0.40},
    'rrf_sbert0p60': {'kind': 'rrf', 'step6_weight': 1.0, 'sbert_weight': 0.60},
    'rrf_equal': {'kind': 'rrf', 'step6_weight': 1.0, 'sbert_weight': 1.0},
    'step6_then_sbert': {'kind': 'append'},
})

IS_KAGGLE = Path('/kaggle/working').exists()
OUTPUT_DIR = Path('/kaggle/working/step6_sbertft') if IS_KAGGLE else Path('task1/pipeline/step6+sbertft/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR.resolve() if not IS_KAGGLE else OUTPUT_DIR)

## Locate input artifact

Local path mặc định là `task1/pipeline/step6+sbertft/step6+sbertft`. Trên Kaggle, notebook scan nhẹ trong `/kaggle/input` để tìm folder có đủ Step 6 và SBERT-FT public rankings.

In [ ]:
REQUIRED_RELATIVE_FILES = [
    Path('public-official.json'),
    Path('step6/rankings/public_rankings_step6_fused.jsonl'),
    Path('step6/submission/submission.json'),
    Path('sbertft/public_submission/public_ranked_contexts.csv'),
    Path('sbertft/public_submission/submission.json'),
]

def has_required_files(root: Path) -> bool:
    return all((root / rel).exists() for rel in REQUIRED_RELATIVE_FILES)

def locate_input_root() -> Path:
    env_root = os.environ.get('STEP6_SBERTFT_INPUT_ROOT')
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        Path('task1/pipeline/step6+sbertft/step6+sbertft'),
        Path('step6+sbertft'),
        Path('/kaggle/input/step6-sbertft/step6+sbertft'),
        Path('/kaggle/input/step6-sbertft'),
    ])
    for candidate in candidates:
        if has_required_files(candidate):
            return candidate
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for candidate in kaggle_input.glob('*'):
            nested = candidate / 'step6+sbertft'
            if has_required_files(nested):
                return nested
            if has_required_files(candidate):
                return candidate
    checked = '\n'.join(str(p) for p in candidates)
    raise FileNotFoundError(f'Cannot locate step6+sbertft input root. Checked:\n{checked}')

INPUT_ROOT = locate_input_root()
print('INPUT_ROOT =', INPUT_ROOT)

## Load rankings

In [ ]:
def read_json(path: Path) -> Any:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def unique_keep_order(items: list[str]) -> list[str]:
    seen = set()
    out = []
    for item in items:
        item = str(item)
        if item and item not in seen:
            seen.add(item)
            out.append(item)
    return out

def load_step6_rankings(path: Path) -> dict[str, list[str]]:
    rankings = {}
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            qid = str(row['query_id'])
            docs = row.get('fused_doc_ids') or row.get('base_doc_ids') or row.get('doc_ids') or []
            rankings[qid] = unique_keep_order([str(x) for x in docs])
    return rankings

def load_sbert_rankings(csv_path: Path, submission_path: Path) -> dict[str, list[str]]:
    rankings = {}
    with csv_path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            qid = str(row['qid'])
            top20 = json.loads(row.get('top20_rerank') or '[]')
            answer = json.loads(row.get('answer') or '[]')
            rankings[qid] = unique_keep_order([str(x) for x in top20] + [str(x) for x in answer])
    submission = read_json(submission_path)
    for qid, item in submission.items():
        answer = item.get('answer', []) if isinstance(item, dict) else item
        rankings.setdefault(str(qid), unique_keep_order([str(x) for x in answer]))
    return rankings

def public_qids(path: Path) -> list[str]:
    obj = read_json(path)
    if isinstance(obj, dict):
        return [str(x) for x in obj.keys()]
    if isinstance(obj, list):
        return [str(item.get('id', item.get('qid', idx))) for idx, item in enumerate(obj)]
    raise TypeError(type(obj))

PUBLIC_QIDS = public_qids(INPUT_ROOT / 'public-official.json')
STEP6_RANKINGS = load_step6_rankings(INPUT_ROOT / 'step6/rankings/public_rankings_step6_fused.jsonl')
SBERT_RANKINGS = load_sbert_rankings(
    INPUT_ROOT / 'sbertft/public_submission/public_ranked_contexts.csv',
    INPUT_ROOT / 'sbertft/public_submission/submission.json',
)

print('public qids:', len(PUBLIC_QIDS))
print('step6 rankings:', len(STEP6_RANKINGS))
print('sbert rankings:', len(SBERT_RANKINGS))
missing_step6 = [qid for qid in PUBLIC_QIDS if qid not in STEP6_RANKINGS]
missing_sbert = [qid for qid in PUBLIC_QIDS if qid not in SBERT_RANKINGS]
print('missing step6:', len(missing_step6))
print('missing sbert:', len(missing_sbert))
assert not missing_step6, missing_step6[:5]
assert not missing_sbert, missing_sbert[:5]

## Fuse and validate

In [ ]:
def weighted_rrf(step6_docs: list[str], sbert_docs: list[str], *, step6_weight: float, sbert_weight: float, rrf_k: int = 60) -> list[str]:
    scores = {}
    first_seen = {}
    for source_idx, (docs, weight) in enumerate([(step6_docs, step6_weight), (sbert_docs, sbert_weight)]):
        for rank, doc_id in enumerate(docs, start=1):
            doc_id = str(doc_id)
            scores[doc_id] = scores.get(doc_id, 0.0) + weight / (rrf_k + rank)
            first_seen.setdefault(doc_id, (source_idx, rank))
    return sorted(scores, key=lambda doc_id: (-scores[doc_id], first_seen[doc_id]))

def append_unique(primary: list[str], secondary: list[str]) -> list[str]:
    return unique_keep_order(primary + secondary)

def build_submission(candidate: dict[str, Any]) -> dict[str, dict[str, list[str]]]:
    submission = {}
    for qid in PUBLIC_QIDS:
        step6_docs = STEP6_RANKINGS[qid]
        sbert_docs = SBERT_RANKINGS[qid]
        if candidate['kind'] == 'append':
            fused = append_unique(step6_docs, sbert_docs)
        else:
            fused = weighted_rrf(
                step6_docs,
                sbert_docs,
                step6_weight=float(candidate['step6_weight']),
                sbert_weight=float(candidate['sbert_weight']),
                rrf_k=RRF_K,
            )
        submission[qid] = {'answer': fused[:MAX_SUBMISSION_DOCS]}
    return submission

def validate_submission(submission: dict[str, Any]) -> dict[str, Any]:
    issues = []
    if set(submission.keys()) != set(PUBLIC_QIDS):
        issues.append({
            'type': 'query_id_mismatch',
            'missing': sorted(set(PUBLIC_QIDS) - set(submission.keys()))[:10],
            'extra': sorted(set(submission.keys()) - set(PUBLIC_QIDS))[:10],
        })
    length_dist = {}
    for qid in PUBLIC_QIDS:
        item = submission.get(qid, {})
        answer = item.get('answer') if isinstance(item, dict) else None
        if not isinstance(answer, list):
            issues.append({'type': 'answer_not_list', 'qid': qid})
            continue
        length_dist[str(len(answer))] = length_dist.get(str(len(answer)), 0) + 1
        if not (1 <= len(answer) <= MAX_SUBMISSION_DOCS):
            issues.append({'type': 'bad_answer_length', 'qid': qid, 'length': len(answer)})
        if len(set(answer)) != len(answer):
            issues.append({'type': 'duplicate_doc_id', 'qid': qid})
        if any(not isinstance(x, str) for x in answer):
            issues.append({'type': 'non_string_doc_id', 'qid': qid})
    return {
        'num_public_queries': len(PUBLIC_QIDS),
        'num_submission_queries': len(submission),
        'answer_length_distribution': dict(sorted(length_dist.items())),
        'num_errors': len(issues),
        'issues': issues[:50],
    }

def zip_submission(submission_json: Path, submission_zip: Path) -> None:
    submission_zip.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(submission_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(submission_json, arcname='submission.json')
    with zipfile.ZipFile(submission_zip, 'r') as zf:
        names = zf.namelist()
    assert names == ['submission.json'], names

step6_only = read_json(INPUT_ROOT / 'step6/submission/submission.json')
sbert_only = read_json(INPUT_ROOT / 'sbertft/public_submission/submission.json')
audit_rows = []

for name, candidate in FUSION_CANDIDATES.items():
    candidate_dir = OUTPUT_DIR / 'candidates' / name
    submission = build_submission(candidate)
    validation = validate_submission(submission)
    assert validation['num_errors'] == 0, validation
    submission_json = candidate_dir / 'submission.json'
    submission_zip = candidate_dir / 'submission.zip'
    write_json(submission_json, submission)
    write_json(candidate_dir / 'submission_validation.json', validation)
    zip_submission(submission_json, submission_zip)

    changed_vs_step6 = sum(submission[qid]['answer'] != step6_only[qid]['answer'] for qid in PUBLIC_QIDS)
    changed_vs_sbert = sum(submission[qid]['answer'] != sbert_only[qid]['answer'] for qid in PUBLIC_QIDS)
    avg_top5_overlap_step6 = sum(len(set(submission[qid]['answer']) & set(step6_only[qid]['answer'])) for qid in PUBLIC_QIDS) / len(PUBLIC_QIDS)
    avg_top5_overlap_sbert = sum(len(set(submission[qid]['answer']) & set(sbert_only[qid]['answer'])) for qid in PUBLIC_QIDS) / len(PUBLIC_QIDS)
    audit_rows.append({
        'candidate': name,
        **candidate,
        'changed_queries_vs_step6': changed_vs_step6,
        'changed_queries_vs_sbert': changed_vs_sbert,
        'avg_top5_overlap_step6': avg_top5_overlap_step6,
        'avg_top5_overlap_sbert': avg_top5_overlap_sbert,
        'submission_zip': str(submission_zip),
    })

default_dir = OUTPUT_DIR / 'candidates' / DEFAULT_CANDIDATE
shutil.copy2(default_dir / 'submission.json', OUTPUT_DIR / 'submission.json')
shutil.copy2(default_dir / 'submission.zip', OUTPUT_DIR / 'submission.zip')
shutil.copy2(default_dir / 'submission_validation.json', OUTPUT_DIR / 'submission_validation.json')

run_report = {
    'status': 'ok',
    'method': 'quick_public_rank_fusion_no_training_no_model_inference',
    'default_candidate': DEFAULT_CANDIDATE,
    'rrf_k': RRF_K,
    'max_submission_docs': MAX_SUBMISSION_DOCS,
    'input_root': str(INPUT_ROOT),
    'candidate_audit': audit_rows,
    'baseline_note': 'rrf_sbert0p60 is the current best public baseline after Codabench score: precision=0.19740000000000005, recall=0.9173333333333332.',
}
write_json(OUTPUT_DIR / 'run_report.json', run_report)
print(json.dumps(run_report, ensure_ascii=False, indent=2))
print('DEFAULT SUBMISSION ZIP:', OUTPUT_DIR / 'submission.zip')